# DIR bank prep (mic impulse responses)

Build a reusable **device impulse response** bank for waveform convolution:

- Source: `training/noises/dirs` (short mic/DIR wavs @ 44.1 kHz).
- Preprocess: mono, resample to 48 kHz, trim leading silence (same policy as RIR).
- Writes npz under `features/dir/` (variable-length IRs as object array).
- No chord audio here — see `mix-dir-cqt.ipynb`.


## Config


In [1]:
from os import path
import pandas as pd
from IPython.display import display

DIR_SOURCE_DIR = path.normpath("../../noises/dirs")
FEATURES_DIR = path.normpath("../../features/dir")
BANK_TAG = "mic6"
FEATURES_OUT = path.join(FEATURES_DIR, f"dirs-{BANK_TAG}.npz")
MANIFEST_OUT = path.join(FEATURES_DIR, f"dirs-{BANK_TAG}.manifest.csv")

FORCE_REBUILD = False
TARGET_SR = 48_000
TRIM_LEADING_DB = -40.0

config_df = pd.DataFrame([
    {"key": "dir_source_dir", "value": DIR_SOURCE_DIR},
    {"key": "features_out", "value": FEATURES_OUT},
    {"key": "manifest_out", "value": MANIFEST_OUT},
    {"key": "bank_tag", "value": BANK_TAG},
    {"key": "force_rebuild", "value": FORCE_REBUILD},
    {"key": "target_sr", "value": TARGET_SR},
    {"key": "trim_leading_db", "value": TRIM_LEADING_DB},
]).set_index("key")
display(config_df)


,value
key,
dir_source_dir,../../noises/dirs
features_out,../../features/dir/dirs-mic6.npz
manifest_out,../../features/dir/dirs-mic6.manifest.csv
bank_tag,mic6
force_rebuild,False
target_sr,48000
trim_leading_db,-40.0


## Select DIRs


In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

dir_paths = sorted(Path(DIR_SOURCE_DIR).glob("*.wav"))
assert dir_paths, f"No wavs in {DIR_SOURCE_DIR}"

rows = []
for p in dir_paths:
    stem = p.stem
    device = stem[3:] if stem.startswith("IR_") else stem
    rows.append({
        "filename": p.name,
        "path": str(p),
        "device": device,
        "keep": 1,
    })

selected_df = pd.DataFrame(rows)
display(pd.DataFrame([{"n_dirs": len(selected_df), "devices": list(selected_df["device"])}]))
display(selected_df)
assert len(selected_df) > 0


,n_dirs,devices
0,6,"[AKGD12, Crystal, Lomo52A5M, MelodiumRM6, Okta..."


,filename,path,device,keep
0,IR_AKGD12.wav,../../noises/dirs/IR_AKGD12.wav,AKGD12,1
1,IR_Crystal.wav,../../noises/dirs/IR_Crystal.wav,Crystal,1
2,IR_Lomo52A5M.wav,../../noises/dirs/IR_Lomo52A5M.wav,Lomo52A5M,1
3,IR_MelodiumRM6.wav,../../noises/dirs/IR_MelodiumRM6.wav,MelodiumRM6,1
4,IR_OktavaMD57.wav,../../noises/dirs/IR_OktavaMD57.wav,OktavaMD57,1
5,IR_STC4035.wav,../../noises/dirs/IR_STC4035.wav,STC4035,1


## Preprocess + save bank


In [3]:
from os import makedirs
import numpy as np
import librosa
import pandas as pd
from IPython.display import display
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


def trim_leading_silence(h: np.ndarray, thresh_db: float):
    peak = float(np.max(np.abs(h)))
    if peak <= 0:
        return h, 0
    thr = peak * (10.0 ** (thresh_db / 20.0))
    idx = np.where(np.abs(h) >= thr)[0]
    if len(idx) == 0:
        return h, 0
    start = int(idx[0])
    return h[start:], start


def load_and_prep_dir(path_str: str) -> dict:
    y, sr = librosa.load(path_str, sr=None, mono=True)
    y = y.astype(np.float32)
    n_raw = int(len(y))
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
    y, n_trim = trim_leading_silence(y, TRIM_LEADING_DB)
    peak = float(np.max(np.abs(y)))
    if peak > 0:
        y = (y / peak).astype(np.float32)
    return {
        "ir": y,
        "sr_native": int(sr),
        "n_raw": n_raw,
        "n_trim_leading": int(n_trim),
        "n_samples": int(len(y)),
        "duration_s": float(len(y) / TARGET_SR),
        "peak_pre_norm": peak,
    }


makedirs(FEATURES_DIR, exist_ok=True)

if path.isfile(FEATURES_OUT) and not FORCE_REBUILD:
    data = np.load(FEATURES_OUT, allow_pickle=True)
    display(pd.DataFrame([{
        "status": "skipped_exists",
        "features_out": FEATURES_OUT,
        "n_dirs": int(len(data["names"])),
        "target_sr": int(data["target_sr"]) if "target_sr" in data.files else None,
        "bank_tag": str(data["bank_tag"]) if "bank_tag" in data.files else "",
    }]))
else:
    irs, names, devices = [], [], []
    n_samples, durations_s, n_trim_leading, sr_native = [], [], [], []
    errors = []

    for row in tqdm(selected_df.to_dict("records"), desc="Prep DIRs"):
        try:
            meta = load_and_prep_dir(row["path"])
        except Exception as exc:
            errors.append({"filename": row["filename"], "error": str(exc)})
            continue
        irs.append(meta["ir"])
        names.append(row["filename"])
        devices.append(row["device"])
        n_samples.append(meta["n_samples"])
        durations_s.append(meta["duration_s"])
        n_trim_leading.append(meta["n_trim_leading"])
        sr_native.append(meta["sr_native"])

    assert irs, "No DIRs preprocessed"
    ir_obj = np.empty(len(irs), dtype=object)
    for i, arr in enumerate(irs):
        ir_obj[i] = arr

    payload = {
        "irs": ir_obj,
        "names": np.asarray(names),
        "devices": np.asarray(devices),
        "rooms": np.asarray(devices),
        "n_samples": np.asarray(n_samples, dtype=np.int32),
        "durations_s": np.asarray(durations_s, dtype=np.float32),
        "n_trim_leading": np.asarray(n_trim_leading, dtype=np.int32),
        "sr_native": np.asarray(sr_native, dtype=np.int32),
        "target_sr": np.asarray(TARGET_SR),
        "trim_leading_db": np.asarray(TRIM_LEADING_DB, dtype=np.float32),
        "bank_tag": np.asarray(BANK_TAG),
        "source_dir": np.asarray(DIR_SOURCE_DIR),
    }
    np.savez_compressed(FEATURES_OUT, **payload)

    man = selected_df[selected_df["filename"].isin(names)].reset_index(drop=True)
    meta_map = {n: i for i, n in enumerate(names)}
    man["bank_index"] = man["filename"].map(meta_map)
    man["n_samples"] = man["bank_index"].map(lambda i: n_samples[i])
    man["duration_s"] = man["bank_index"].map(lambda i: durations_s[i])
    man["n_trim_leading"] = man["bank_index"].map(lambda i: n_trim_leading[i])
    man["sr_native"] = man["bank_index"].map(lambda i: sr_native[i])
    man.to_csv(MANIFEST_OUT, index=False)

    display(pd.DataFrame([{
        "status": "saved",
        "features_out": FEATURES_OUT,
        "manifest_out": MANIFEST_OUT,
        "n_dirs": len(names),
        "duration_s_min": float(np.min(durations_s)),
        "duration_s_median": float(np.median(durations_s)),
        "duration_s_max": float(np.max(durations_s)),
        "n_errors": len(errors),
    }]))
    if errors:
        display(pd.DataFrame(errors))


Prep DIRs:   0%|          | 0/6 [00:00<?, ?it/s]

,status,features_out,manifest_out,n_dirs,duration_s_min,duration_s_median,duration_s_max,n_errors
0,saved,../../features/dir/dirs-mic6.npz,../../features/dir/dirs-mic6.manifest.csv,6,0.500021,0.500021,0.500021,0


## Verify


In [4]:
import numpy as np
import pandas as pd
from IPython.display import display

data = np.load(FEATURES_OUT, allow_pickle=True)
irs = data["irs"]
names = data["names"].astype(str)
devices = data["devices"].astype(str)
durs = data["durations_s"].astype(np.float32)

assert int(data["target_sr"]) == TARGET_SR
assert all(isinstance(irs[i], np.ndarray) for i in range(len(irs)))
peaks = [float(np.max(np.abs(irs[i]))) for i in range(len(irs))]

display(pd.DataFrame([{
    "n_dirs": len(names),
    "devices": list(devices),
    "target_sr": int(data["target_sr"]),
    "duration_s_min": float(durs.min()),
    "duration_s_max": float(durs.max()),
    "peak_min": float(np.min(peaks)),
    "peak_max": float(np.max(peaks)),
    "bank_tag": str(data["bank_tag"]),
}]))


,n_dirs,devices,target_sr,duration_s_min,duration_s_max,peak_min,peak_max,bank_tag
0,6,"[AKGD12, Crystal, Lomo52A5M, MelodiumRM6, Okta...",48000,0.500021,0.500021,1.0,1.0,mic6
